In [15]:
from openai import OpenAI
client = OpenAI(
   api_key="sk-9833dc6f7f814ae29b743d4003e49e56",
   base_url="https://api.deepseek.com/v1"
)

In [16]:
# 文件型的关系型数据库
import sqlite3
conn = sqlite3.connect("test.db")
# 创建一个游标对象
# 通过 光标（cursor） 执行sql操作
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    department TEXT,
    salary INTEGER
)
""")
sample_data = [
    (6, "张三", "销售", 5000),
    (7, "李四", "市场", 6000),
    (8, "王五", "研发", 7000),
    (9, "赵六", "工程", 8000),
    (10, "王二", "市场", 9000),
]
cursor.executemany(
    "INSERT INTO employees VALUES (?,?,?,?)", 
    sample_data
)
conn.commit()

In [17]:
# 获取数据库Schema
# Schema 应详细描述每个表的字段和类型，
# 这有助于 Claude 理解表的结构和关联。
# SQLite命令，查看employees表的列名、类型等结构信息。
schema = cursor.execute("PRAGMA table_info(employees)").fetchall()
print(schema)
# 列表推导式
# 使用 f-string（格式化字符串字面量），将列名和类型拼接成一个字符串。
schema_str = "CREATE TABLE EMPLOYEES (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
print("数据库Schema:")
print(schema_str)

## 销售部门平均工资多少？先分组再算平均
# text2sql 数据库平权 
# vibe coding 平权了代码开发 
def ask_deepseek(query, schema):
    # 模版
    prompt = f"""
    这是一个数据库的Schema：
    {schema}
    根据这个Schema, 请输出一个SQL查询来回答以下问题。
    只输出SQL 查询语句本身，不要使用任何Markdown格式，
    不要包含反引号、代码块标记或额外说明。
    问题：{query}
    """
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        max_tokens=2048,
        messages=[{
            "role": "user",
            "content": prompt
        }]
    )
    return response.choices[0].message.content
    

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'name', 'TEXT', 0, None, 0), (2, 'department', 'TEXT', 0, None, 0), (3, 'salary', 'INTEGER', 0, None, 0)]
数据库Schema:
CREATE TABLE EMPLOYEES (
id INTEGER
name TEXT
department TEXT
salary INTEGER
)


In [18]:
question = "工程部门员工的姓名和工资是多少？"
sql_query = ask_deepseek(question, schema_str)
print(sql_query)

SELECT name, salary FROM EMPLOYEES WHERE department = '工程';


In [19]:
results = cursor.execute(sql_query).fetchall()
for row in results:
    print(row)

('赵六', 8000)


In [21]:
question = "在销售部门的增加一个新员工， 姓名为张三，工资为45000"
sql_query1 = ask_deepseek(question, schema_str)
print(sql_query1)
cursor.execute(sql_query1)
conn.commit()

INSERT INTO EMPLOYEES (name, department, salary) VALUES ('张三', '销售', 45000);


In [22]:
question = "将王二的工资改成55000"
sql_query2 = ask_deepseek(question, schema_str)
print(sql_query2)
cursor.execute(sql_query2)
conn.commit()

UPDATE EMPLOYEES SET salary = 55000 WHERE name = '王二';


In [26]:
question = "查询所有员工的信息"
sql_query3 = ask_deepseek(question, schema_str)
print(sql_query3)
result2 = cursor.execute(sql_query3).fetchall()
for row in result2:
    print(row)

SELECT * FROM EMPLOYEES;
(6, '张三', '销售', 5000)
(7, '李四', '市场', 6000)
(8, '王五', '研发', 7000)
(9, '赵六', '工程', 8000)
(10, '王二', '市场', 55000)
(11, '张三', '销售', 45000)
